# Multivariate differentiation (Figure 5) from dandiset 000036

Part of the attempt on Mayner, Marshall, Billeh et al. (2022), eNeuro 9(1)
ENEURO.0280-21.2021 ([doi:10.1523/eneuro.0280-21.2021](https://doi.org/10.1523/eneuro.0280-21.2021)),
on [DANDI:000036](https://dandiarchive.org/dandiset/000036/0.230515.1917) version 0.230515.1917.
The header of `reproduce_mayner_2022.ipynb` says why this version and gives the table of all
claims; the data check there applies here. This notebook tests the two claims that need UMAP
(fig5-multivariate-layer and fig5-multivariate-area), which take too long to share a notebook
with the rest. Sessions are the 44 the paper analysed, found through the link claim
(session-link) of the main notebook.

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMBA_NUM_THREADS"] = "1"

import multiprocessing as mp
import time
import warnings
from concurrent.futures import ProcessPoolExecutor
from importlib.metadata import version

import numpy as np
import pandas as pd

import openscope_nd as o

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

started = time.time()
assets = o.list_assets()
meta = o.read_all_metadata(assets)
paper = pd.read_parquet(o.CODE_RESULTS + "main.parquet")
link = o.link_to_paper(meta, paper)
use = link[link.in_paper].reset_index(drop=True)
print(f"dandiset {o.DANDISET_ID} version {o.DANDISET_VERSION}; {len(use)} of {len(link)} sessions linked to the paper")
print("umap-learn", version("umap-learn"), "; numba", version("numba"), "; UMAP parameters:", o.UMAP_PARAMS)

## Data check

In [ ]:
print("Linked sessions by layer and area:")
print(pd.crosstab(use.layer, use.area).reindex(index=o.LAYERS, columns=o.AREAS).to_string())
print("Paper's area and layer agree for every linked session:",
      bool((use.area == use.paper_area.astype(str)).all() and (use.layer == use.paper_layer.astype(str)).all()))

In [ ]:
t0 = time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context("spawn")) as ex:
    out = list(ex.map(o.umap_session, use.url))
print(f"UMAP on {len(out)} sessions in {(time.time() - t0) / 60:.1f} min")
mcd = pd.concat([r["trials"].assign(session=s) for r, s in zip(out, use.identifier)])
mcd = mcd.merge(use[["identifier", "layer", "area", "mouse_id"]], left_on="session", right_on="identifier")
mcd = mcd.merge(o.STIMULI.reset_index()[["stimulus", "stimulus_type"]], on="stimulus")
mcd["y"] = np.log10(mcd.mcd)

In [ ]:
fr = pd.Series({s: r["n_frames"] for r, s in zip(out, use.identifier)})
print(f"Frames embedded per session: {fr.min()} to {fr.max()}; cells per session: "
      f"{min(r['ncells'] for r in out)} to {max(r['ncells'] for r in out)}")
print("Trials:", len(mcd), "; per session:", mcd.groupby("session").size().unique(),
      "; frames per trial:", mcd.n_frames.min(), "to", mcd.n_frames.max())
print(f"log10 mean centroid distance: {mcd.y.min():.3f} to {mcd.y.max():.3f}; non-finite: {int((~np.isfinite(mcd.y)).sum())}")
ref = pd.read_parquet(o.CODE_RESULTS + "mean_centroid_distance.parquet")
ref = ref[ref.stimulus_is_scrambled_pair].assign(stimulus=lambda d: d.stimulus.astype(str), trial=lambda d: d.trial.astype(int))
cmp = mcd.merge(use[["identifier", "paper_session"]], left_on="session", right_on="identifier").merge(
    ref[["session", "stimulus", "trial", "log(mean_centroid_distance)"]],
    left_on=["paper_session", "stimulus", "trial"], right_on=["session", "stimulus", "trial"])
print(f"Against the paper's per-trial values (results/mean_centroid_distance.parquet): n = {len(cmp)}, "
      f"r = {np.corrcoef(cmp.y, cmp['log(mean_centroid_distance)'])[0, 1]:.3f}, "
      f"mean difference {(cmp.y - cmp['log(mean_centroid_distance)']).mean():+.4f}")

UMAP is stochastic and its result depends on the library version; the paper's per-trial
values come from its own run (umap-learn version not pinned in the code's environment files
beyond what they list). The comparison with the paper's values is a reference, not a test.

## fig5-multivariate-layer

> Multivariate ND (mean distance to the centroid in an 8-dimensional UMAP embedding) is
> higher for unscrambled than scrambled stimuli specifically in L2/3.

In [ ]:
def show(r, label):
    print(f"{label}: interaction chi2({r['df']}) = {r['chi2']:.3f}, p = {r['p']:.3g} ({r['n']} trials, {r['sessions']} sessions)")
    print(r["posthoc"].round(4).to_string())


r5l = o.category_models(mcd, "layer")
show(r5l, "layer")
d_layer = mcd.groupby("layer").apply(lambda g: o.cohens_d(g.y, g.stimulus_type)).reindex(o.LAYERS)
print("Cohen's d per layer:", d_layer.round(3).to_dict(), "(paper L2/3 0.194, L4 0.0994, L5 -0.0651)")
print("Cohen's d per layer and mouse:")
print(mcd.groupby(["layer", "mouse_id"]).apply(lambda g: o.cohens_d(g.y, g.stimulus_type)).round(3).to_string())

**Result: reproduced.** The layer by category interaction on log10 mean centroid distance is
chi2(2) = 18.99, p = 7.5e-5 (paper 18.135, within tolerance). L2/3 is significant (z = 5.19,
adjusted p < 0.0001, Cohen's d 0.194 against the paper's 0.194) and L4 (p = 0.107) and L5 (p =
0.997) are not. Within L2/3 the effect is positive in all three mice (d 0.05, 0.79 and 0.25).

## fig5-multivariate-area

> Multivariate ND is higher for unscrambled than scrambled stimuli specifically in AL and AM.

In [ ]:
r5a = o.category_models(mcd, "area")
show(r5a, "area")
d_area = mcd.groupby("area").apply(lambda g: o.cohens_d(g.y, g.stimulus_type)).reindex(o.AREAS)
print("Cohen's d per area:", d_area.round(3).to_dict(), "(paper AL 0.184, AM 0.277)")
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")

**Result: reproduced.** The area interaction is chi2(4) = 16.20, p = 0.0028 (paper 16.232); AL
(adjusted p = 0.0098) and AM (p < 0.0001) are significant and V1, LM and PM are not (p >=
0.92). Cohen's d is 0.181 for AL and 0.275 for AM (paper 0.184 and 0.277), within tolerance.

## Summary

| Claim | Outcome |
|---|---|
| fig5-multivariate-layer | reproduced |
| fig5-multivariate-area | reproduced |

The table of all claims, the differences from the paper's method and what was found about the
dataset are in the summary of `reproduce_mayner_2022.ipynb`. Here UMAP is run with umap-learn
(version printed above), whose embeddings differ from the paper's run; the per-trial values
correlate with the paper's at r = 0.995, and the conclusions do not depend on the difference.